In [0]:
dbutils.widgets.text("catalog","claims")
catalog = dbutils.widgets.get("catalog")

from pyspark.sql import functions as F, Window
from delta.tables import DeltaTable

In [0]:
spark.sql(f"USE CATALOG {catalog}")

In [0]:
spark.sql("create table if not exists bronze.policy")

In [0]:
spark.sql(f"""
copy into bronze.policy
from "/Volumes/{catalog}/raw/landing/policy_changes_cdc/"
FILEFORMAT = JSON
format_options ('inferSchema' = 'true')
copy_options ('mergeSchema' = 'true')
""")

In [0]:
df = spark.read.table("bronze.policy")


In [0]:
df.display()

In [0]:
df = df.withColumns({
    "effective_from" : F.to_date(F.col("effective_from"), "yyyy-MM-dd"),
    "op_ts" : F.to_timestamp(F.col("op_ts"), "yyyy-MM-dd'T'HH:mm:ss"),
})

In [0]:
w = Window.partitionBy(F.col("member_id"), F.col("policy_id"), F.col("effective_from")).orderBy(F.col("op_ts").desc())

dedup = df.withColumn("rank", F.row_number().over(w)) \
    .filter(F.col("rank") == 1) \
        .drop("rank")
dedup.display()

In [0]:
w = Window.partitionBy(F.col("member_id")).orderBy("effective_from", "op_ts")

staged = dedup.withColumn("effective_till", F.lead("effective_from", 1).over(w)) \
    .withColumn("is_current", F.col("effective_till").isNull() & (F.col("op") != "DELETE")) \
    .filter(F.col("op") != "DELETE") \
        .select("member_id", "policy_id", "plan_code", "op", "op_ts", "effective_from", "effective_till", "is_current") 
staged.display()

In [0]:
if not spark.catalog.tableExists("silver.dim_policy_scd2"):
    DeltaTable.create(spark) \
        .tableName("silver.dim_policy_scd2") \
        .addColumns(staged.schema) \
        .execute()

In [0]:
target = DeltaTable.forName(spark, "silver.dim_policy_scd2").alias("target")

target.merge(
    staged.alias("source"),
    """
    target.member_id = source.member_id
    AND target.policy_id = source.policy_id
    AND target.effective_from = source.effective_from
    """
) \
    .whenMatchedUpdate(set = {
        "plan_code": "source.plan_code",
        "effective_till": "source.effective_till",
        "is_current": "source.is_current"
    }) \
    .whenNotMatchedInsert(values= {
        "member_id" : "source.member_id",
        "policy_id" : "source.policy_id",
        "plan_code" : "source.plan_code",
        "effective_from" : "source.effective_from",
        "effective_till" : "source.effective_till",
        "is_current" : "source.is_current"
    }) \
    .execute()